In [18]:
import pandas as pd

# Load dataset
df = pd.read_csv("legal_bias_dataset.csv")

# Display first few rows
df.head()

,Case_ID,Case_Type,Court_Name,Judges,Case_Facts,Legal_Reasoning,Verdict,Judicial_Opinions,Bias_Indicative_Sentences,Legal_Citations,Document_Length,Sentiment_Polarity_Score,Bias_Label
0,CASE_0001,Administrative,District Court,Single Judge,Last establish hot left perhaps perform withou...,Produce develop person quality only wish prese...,Guilty,Case student theory scene particular yeah. Wai...,NaN,Section 348 of Law Act 1970,2564,Neutral,0
1,CASE_0002,Others,District Court,Single Judge,Government shake loss the home word while. Cel...,Hair him its position agency write. Yard stand...,Dismissed,Treatment project woman suffer require. From e...,Financial build bad on defense.,Section 488 of Law Act 1953,2025,Negative,1
2,CASE_0003,Civil,District Court,Multiple Judges,Finally also begin base production exactly par...,Home since leave girl offer organization final...,Acquitted,Media yet food could property. Pull family you...,Team answer drug.,Section 74 of Law Act 2006,2035,Neutral,1
3,CASE_0004,Administrative,High Court,Multiple Judges,Recent boy coach mention identify age. Account...,More second reduce to church different. Mr hot...,Compensation Granted,Strong present career trade. Job knowledge bit...,Beat condition job camera position force.,Section 18 of Law Act 1966,1829,Neutral,0
4,CASE_0005,Administrative,Magistrate Court,Single Judge,Thousand crime policy left send. Yeah intervie...,Type institution truth phone. Inside citizen f...,Guilty,Fly new sport foreign enjoy example her. Repub...,NaN,Section 35 of Law Act 1980,3471,Positive,1


In [19]:
# Check dataset size and missing values
print("Rows:", len(df))
print(df.isna().sum())

# Show column names
print("\nColumns:", df.columns.tolist())

Rows: 1500
Case_ID                        0
Case_Type                      0
Court_Name                     0
Judges                         0
Case_Facts                     0
Legal_Reasoning                0
Verdict                        0
Judicial_Opinions              0
Bias_Indicative_Sentences    766
Legal_Citations                0
Document_Length                0
Sentiment_Polarity_Score       0
Bias_Label                     0
dtype: int64

Columns: ['Case_ID', 'Case_Type', 'Court_Name', 'Judges', 'Case_Facts', 'Legal_Reasoning', 'Verdict', 'Judicial_Opinions', 'Bias_Indicative_Sentences', 'Legal_Citations', 'Document_Length', 'Sentiment_Polarity_Score', 'Bias_Label']


In [20]:
import spacy
nlp = spacy.load("en_core_web_sm")
print("SpaCy loaded successfully!")

SpaCy loaded successfully!


In [21]:
print(df.columns)

Index(['Case_ID', 'Case_Type', 'Court_Name', 'Judges', 'Case_Facts',
       'Legal_Reasoning', 'Verdict', 'Judicial_Opinions',
       'Bias_Indicative_Sentences', 'Legal_Citations', 'Document_Length',
       'Sentiment_Polarity_Score', 'Bias_Label'],
      dtype='object')


In [22]:
df['clean_text'] = df['Case_Facts'].astype(str)

In [34]:
print(df[['Case_Facts', 'clean_text']].head())

                                          Case_Facts  \
0  Last establish hot left perhaps perform withou...   
1  Government shake loss the home word while. Cel...   
2  Finally also begin base production exactly par...   
3  Recent boy coach mention identify age. Account...   
4  Thousand crime policy left send. Yeah intervie...   

                                          clean_text  
0  Last establish hot left perhaps perform withou...  
1  Government shake loss the home word while. Cel...  
2  Finally also begin base production exactly par...  
3  Recent boy coach mention identify age. Account...  
4  Thousand crime policy left send. Yeah intervie...  


In [35]:
import spacy

# Load small English model
nlp = spacy.load("en_core_web_sm")

# Define a function to clean text
def clean_with_spacy(text):
    doc = nlp(text)
    cleaned_tokens = [token.lemma_.lower() for token in doc 
                      if not token.is_stop and token.is_alpha]
    return " ".join(cleaned_tokens)

# Apply cleaning function
df['clean_text'] = df['clean_text'].apply(clean_with_spacy)

# Preview result
print(df[['Case_Facts', 'clean_text']].head())

                                          Case_Facts  \
0  Last establish hot left perhaps perform withou...   
1  Government shake loss the home word while. Cel...   
2  Finally also begin base production exactly par...   
3  Recent boy coach mention identify age. Account...   
4  Thousand crime policy left send. Yeah intervie...   

                                          clean_text  
0  establish hot left perform government read for...  
1  government shake loss home word cell state wor...  
2  finally begin base production exactly particul...  
3  recent boy coach mention identify age account ...  
4  thousand crime policy left send yeah interview...  


In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000)  # limit to top 5000 words

# Fit and transform clean_text column
X_tfidf = tfidf.fit_transform(df['clean_text'])

print("TF-IDF shape:", X_tfidf.shape)  # (num_rows, num_features)

TF-IDF shape: (1500, 776)


In [37]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Example target variable: df['Bias_Label']
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, df['Bias_Label'], test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.5133333333333333


In [38]:
pip install gensim

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [39]:
import sys
print(sys.executable)

C:\ProgramData\anaconda3\python.exe


In [40]:
!{sys.executable} -m pip install gensim

Defaulting to user installation because normal site-packages is not writeable


In [41]:
from gensim.models import Word2Vec
import numpy as np

In [46]:
from gensim.models import Word2Vec
import numpy as np

# Tokenize each text into list of words
tokenized_texts = [text.split() for text in df['clean_text']]

# Train Word2Vec model
w2v_model = Word2Vec(sentences=tokenized_texts, vector_size=100, window=5, min_count=2, workers=4)

# Get average word vector per sentence
def get_vector(text):
    words = text.split()
    vectors = [w2v_model.wv[word] for word in words if word in w2v_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

df['w2v_vector'] = df['clean_text'].apply(get_vector)

In [47]:
df['w2v_vector'].head()

0    [-0.10242013, 0.20750546, 0.092659764, 0.05302...
1    [-0.112360924, 0.22155252, 0.098411635, 0.0559...
2    [-0.113461934, 0.22374684, 0.101095274, 0.0584...
3    [-0.11176372, 0.21351358, 0.096367806, 0.05585...
4    [-0.105313346, 0.20560707, 0.091409564, 0.0533...
Name: w2v_vector, dtype: object

In [48]:

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Step 3: Generate sentence embeddings
bert_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = bert_model.encode(df['clean_text'], show_progress_bar=True)

# Step 4: Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, df['Verdict'], test_size=0.2, random_state=42
)

# Step 5: Train Logistic Regression model
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

# Step 6: Predict and evaluate
y_pred = log_reg.predict(X_test)

print("✅ Logistic Regression Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

✅ Logistic Regression Accuracy: 0.18666666666666668

Classification Report:
                       precision    recall  f1-score   support

           Acquitted       0.19      0.16      0.17        62
      Appeal Allowed       0.23      0.24      0.24        62
Compensation Granted       0.13      0.19      0.15        52
           Dismissed       0.18      0.12      0.15        64
              Guilty       0.22      0.22      0.22        60

            accuracy                           0.19       300
           macro avg       0.19      0.19      0.19       300
        weighted avg       0.19      0.19      0.19       300



In [49]:
from sklearn.svm import SVC

# Train SVM model
svm_model = SVC(kernel='linear')
svm_model.fit(X_train, y_train)

# Predict
y_pred_svm = svm_model.predict(X_test)

# Evaluate
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.19
                      precision    recall  f1-score   support

           Acquitted       0.20      0.18      0.19        62
      Appeal Allowed       0.19      0.19      0.19        62
Compensation Granted       0.16      0.27      0.20        52
           Dismissed       0.16      0.09      0.12        64
              Guilty       0.24      0.23      0.24        60

            accuracy                           0.19       300
           macro avg       0.19      0.19      0.19       300
        weighted avg       0.19      0.19      0.19       300



In [52]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1200, 384)
X_test shape: (300, 384)
